# Etapa 4 — Aprendizado probabilístico e EM

**Primeira entrega:** MLE, MAP e Naive Bayes próprio. O EM e sua comparação serão incorporados posteriormente.

**Protocolo:** usar apenas desenvolvimento/2024, validação espacial em cinco folds e transformação calculada no treino. O conjunto de 2025 permanece reservado para avaliação final.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils import carregar_dados, iterar_folds, preparar_particao
from aprendizado_probabilistico import mle_bernoulli, map_bernoulli, BernoulliNaiveBayes

## 1. Estimativas MLE e MAP

Para $y_i\in\{0,1\}$, $\hat\theta_{MLE}=\sum_i y_i/n$. Com priori $\mathrm{Beta}(\alpha,\beta)$, a moda posterior interior é $\hat\theta_{MAP}=(k+\alpha-1)/(n+\alpha+\beta-2)$, quando ambos os parâmetros posteriores são maiores que um.

In [ ]:
# Teste verificável com 3 sucessos em 4 observações
y_exemplo = np.array([1, 1, 1, 0])
print('MLE:', mle_bernoulli(y_exemplo))
print('MAP Beta(2,2):', map_bernoulli(y_exemplo, 2, 2))
assert np.isclose(mle_bernoulli(y_exemplo), 0.75)
assert np.isclose(map_bernoulli(y_exemplo, 2, 2), 2/3)

### Aplicação ao domínio

Estimamos a frequência da classe `FRP elevada` apenas com os rótulos do treino. O corte de FRP elevada é o percentil 75 do próprio treino, calculado por `preparar_particao()`. Os folds mantêm o mesmo protocolo do restante da equipe.

In [ ]:
dados = carregar_dados(2024)
resultados_estimadores = []
for fold, treino, validacao in iterar_folds(dados, 'espacial'):
    p = preparar_particao(treino, validacao, 'discretizada')
    y = p.y_cls_treino.to_numpy()
    resultados_estimadores.append({'fold': fold, 'n_treino': len(y), 'limiar_frp': p.limiar_frp,
                                   'positivos': int(y.sum()), 'MLE': mle_bernoulli(y),
                                   'MAP_Beta_2_2': map_bernoulli(y, 2, 2)})
tabela_estimadores = pd.DataFrame(resultados_estimadores)
display(tabela_estimadores)

In [ ]:
# Efeito da priori em amostras pequenas (apenas ilustração, sem usar validação)
y_pequeno = np.array([1, 1, 0, 1, 0])
experimento_priori = pd.DataFrame([{'estimador': 'MLE', 'theta': mle_bernoulli(y_pequeno)},
                                   {'estimador': 'MAP Beta(2,2)', 'theta': map_bernoulli(y_pequeno, 2, 2)},
                                   {'estimador': 'MAP Beta(5,5)', 'theta': map_bernoulli(y_pequeno, 5, 5)}])
display(experimento_priori)

## 2. Naive Bayes de Bernoulli do zero

A implementação em `aprendizado_probabilistico.py` estima as probabilidades condicionais dos atributos binários com suavização aditiva e calcula as probabilidades posteriores no espaço logarítmico. A classe 1 significa FRP superior ou igual ao Q75 do treino.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
linhas = []
for fold, treino, validacao in iterar_folds(dados, 'espacial'):
    p = preparar_particao(treino, validacao, 'discretizada')
    Xtr, Xva = p.X_treino.to_numpy(), p.X_avaliacao.to_numpy()
    ytr, yva = p.y_cls_treino.to_numpy(), p.y_cls_avaliacao.to_numpy()
    modelo = BernoulliNaiveBayes(alpha=1.0).fit(Xtr, ytr)
    pred = modelo.predict(Xva)
    linhas.append({'fold': fold, 'limiar_frp': p.limiar_frp,
                   'acuracia': accuracy_score(yva, pred),
                   'precisao': precision_score(yva, pred, zero_division=0),
                   'recall': recall_score(yva, pred, zero_division=0),
                   'f1': f1_score(yva, pred, zero_division=0),
                   'tn': confusion_matrix(yva, pred, labels=[0,1])[0,0],
                   'fp': confusion_matrix(yva, pred, labels=[0,1])[0,1],
                   'fn': confusion_matrix(yva, pred, labels=[0,1])[1,0],
                   'tp': confusion_matrix(yva, pred, labels=[0,1])[1,1]})
tabela_nb = pd.DataFrame(linhas)
display(tabela_nb)
display(tabela_nb[['acuracia','precisao','recall','f1']].agg(['mean','std']))

## 3. Interpretação preliminar

Preencher após executar o notebook com os dados reais: avaliar a diferença MLE/MAP e comparar a acurácia do NB à referência majoritária (~0,75), sem ignorar precisão, recall e F1.

**Próxima implementação:** `em.py` com misturas gaussianas diagonais para 2 e 3 componentes; gráfico da log-verossimilhança e comparação controlada NB puro vs. NB+EM. A comparação principal com EM deve utilizar a mesma família de distribuição nos dois classificadores.